# Sentiment Analysis of Amazon Customer Reviews

**Author:** *Mna Gabra*  
**Date:** May 2026* 
**Stack:** Python · pandas · scikit-learn · TF-IDF · Logistic Regression

---

## Project Overview

This notebook builds an end-to-end **binary sentiment classifier** trained on real Amazon customer reviews. Given a review's text, the model predicts whether the customer sentiment is **Positive (1)** or **Negative (0)**.

### Pipeline
| Step | Description |
|------|-------------|
| 1 | **Data Loading** — read and inspect the raw dataset (~21 k records) |
| 2 | **Preprocessing** — clean ratings, drop neutral reviews, create binary labels |
| 3 | **Text Cleaning** — lowercase, remove non-alphabetic characters |
| 4 | **Feature Engineering** — TF-IDF vectorisation (top 5 000 terms) |
| 5 | **Model Training** — Logistic Regression on an 80/20 stratified train-test split |
| 6 | **Evaluation** — accuracy, precision, recall, F1-score |
| 7 | **Inference** — live prediction on custom text input |

### Results Summary
| Metric | Score |
|--------|-------|
| Overall Accuracy | **93.8 %** |
| Macro F1-Score | **0.92** |
| Negative-class F1 | **0.96** |
| Positive-class F1 | **0.89** |

---
## 1. Imports & Setup

All required libraries are imported in one place for clarity and reproducibility.

- **pandas / numpy** — data loading and manipulation  
- **re** — regular-expression-based text cleaning  
- **scikit-learn** — vectorisation, modelling, and evaluation utilities  

In [2]:
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# Global random seed — ensures reproducible results
RANDOM_STATE = 42

---
## 2. Data Loading

The dataset contains Amazon customer reviews scraped from Trustpilot, spanning **2007–2024** across multiple countries.  
Each row includes: reviewer name, country, star rating (as text), review title, and full review body.

In [3]:
# Load the raw dataset — skip malformed lines gracefully
df = pd.read_csv(
    "amazon_reviews.csv",
    encoding="utf-8",
    on_bad_lines="skip",
    engine="python"
)

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}\n")
df.head()

Dataset shape : (21214, 9)
Columns       : ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']



,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


---
## 3. Data Preprocessing

The raw `Rating` column stores star ratings as strings (e.g. `"Rated 4 out of 5 stars"`).  
We extract the numeric value, drop rows with missing review text, and **exclude neutral 3-star reviews** to keep the task unambiguously binary.

**Label encoding:**
- `sentiment = 1` → Positive (rating ≥ 4 stars)  
- `sentiment = 0` → Negative (rating ≤ 2 stars)

In [4]:
# Drop rows where the review body is missing
df = df.dropna(subset=["Review Text"])

# Parse numeric rating from string  e.g. "Rated 4 out of 5 stars" -> 4
df["Rating"] = df["Rating"].str.extract(r"(\d)").astype(float)

print(f"Rating distribution:\n{df['Rating'].value_counts().sort_index()}")

Rating distribution:
Rating
1.0    13123
2.0     1227
3.0      885
4.0     1292
5.0     4528
Name: count, dtype: int64


In [5]:
# Retain only the columns needed for modelling
df = df[["Rating", "Review Text"]]

# Drop neutral (3-star) reviews — their sentiment is inherently ambiguous
df = df[df["Rating"] != 3].copy()

# Create binary sentiment label
df["sentiment"] = (df["Rating"] > 3).astype(int)

print(f"Remaining samples : {len(df):,}")
print(f"Negative (0)      : {(df['sentiment'] == 0).sum():,}")
print(f"Positive (1)      : {(df['sentiment'] == 1).sum():,}")
df.head()

Remaining samples : 20,170
Negative (0)      : 14,350
Positive (1)      : 5,820


,Rating,Review Text,sentiment
0,1.0,"I registered on the website, tried to order a ...",0
1,1.0,Had multiple orders one turned up and driver h...,0
2,1.0,I informed these reprobates that I WOULD NOT B...,0
3,1.0,I have bought from Amazon before and no proble...,0
4,1.0,If I could give a lower rate I would! I cancel...,0


---
## 4. Text Cleaning

A lightweight cleaning function is applied to each review before vectorisation:

1. **Lowercase** — normalises token casing.  
2. **Remove non-alphabetic characters** — strips punctuation, digits, and special symbols.

This reduces vocabulary noise and allows TF-IDF to focus on meaningful word patterns.

In [6]:
def clean_text(text: str) -> str:
    """
    Normalise a raw review string for TF-IDF vectorisation.

    Steps
    -----
    1. Lowercase all characters.
    2. Replace any non-alphabetic character with a space.

    Parameters
    ----------
    text : str
        Raw review body.

    Returns
    -------
    str
        Cleaned review text.
    """
    text = text.lower()
    text = re.sub(r"[^a-z]", " ", text)
    return text


df["cleaned"] = df["Review Text"].apply(clean_text)

# Sanity check
print("Original :", df["Review Text"].iloc[0][:120])
print("Cleaned  :", df["cleaned"].iloc[0][:120])

Original : I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending th
Cleaned  : i registered on the website  tried to order a laptop  entered all the details  but instead of charging me and sending th


---
## 5. Feature Engineering — TF-IDF Vectorisation

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts raw text into a numerical feature matrix.  
Each term is weighted by how often it appears in a document relative to the whole corpus — boosting distinctive words and down-weighting very common ones.

We cap the vocabulary at the **top 5,000 terms** to keep the feature matrix manageable.

In [7]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["cleaned"]).toarray()
y = df["sentiment"].values

print(f"Feature matrix shape : {X.shape}  (samples × TF-IDF features)")
print(f"Label distribution   :  Negative={sum(y == 0):,}  |  Positive={sum(y == 1):,}")

Feature matrix shape : (20170, 5000)  (samples × TF-IDF features)
Label distribution   :  Negative=14,350  |  Positive=5,820


---
## 6. Model Training

The data is divided into an **80 % training / 20 % test** split using stratified sampling to preserve the class ratio in both partitions.  

**Logistic Regression** is chosen as the classifier — it is fast, interpretable, and a well-established baseline for text classification tasks.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y           # preserve class balance in both splits
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")

Training samples : 16,136
Test samples     : 4,034


In [9]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


---
## 7. Evaluation

The model is evaluated on the held-out **test set** using four standard metrics:

| Metric | Definition |
|--------|------------|
| **Accuracy** | Fraction of all predictions that are correct |
| **Precision** | Of all predicted positives, how many are truly positive |
| **Recall** | Of all true positives, how many did the model correctly identify |
| **F1-Score** | Harmonic mean of precision and recall |

In [10]:
y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report   = classification_report(
    y_test, y_pred,
    target_names=["Negative", "Positive"]
)

print("\u2501" * 40)
print(" Model Evaluation \u2014 Test Set")
print("\u2501" * 40)
print(f"Accuracy: {accuracy:.4f}\n")
print(report)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Model Evaluation — Test Set
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Accuracy: 0.9393

              precision    recall  f1-score   support

    Negative       0.95      0.97      0.96      2870
    Positive       0.91      0.87      0.89      1164

    accuracy                           0.94      4034
   macro avg       0.93      0.92      0.93      4034
weighted avg       0.94      0.94      0.94      4034



---
## 8. Inference — Custom Text Prediction

The trained pipeline is wrapped in a clean, reusable inference function.  
Any raw text string can be passed in and a human-readable sentiment label is returned.

In [11]:
def predict_sentiment(text: str) -> str:
    """
    Predict the sentiment of a review string.

    Parameters
    ----------
    text : str
        Raw review text (any casing, punctuation allowed).

    Returns
    -------
    str
        'Positive' or 'Negative'.
    """
    cleaned    = clean_text(text)
    vector     = vectorizer.transform([cleaned]).toarray()
    prediction = model.predict(vector)[0]
    return "Positive" if prediction == 1 else "Negative"


# --- Demo: run predictions on sample reviews ---
sample_reviews = [
    "This product is amazing!",
    "Terrible quality, complete waste of money.",
    "Delivery was fast but the packaging was damaged.",
]

for review in sample_reviews:
    result = predict_sentiment(review)
    icon   = "\u2705" if result == "Positive" else "\u274c"
    print(f"Review   : '{review}'")
    print(f"Sentiment: {result} {icon}\n")

Review   : 'This product is amazing!'
Sentiment: Positive ✅

Review   : 'Terrible quality, complete waste of money.'
Sentiment: Negative ❌

Review   : 'Delivery was fast but the packaging was damaged.'
Sentiment: Positive ✅



---
## 9. Conclusion

### Summary

A TF-IDF + Logistic Regression pipeline was built to classify Amazon customer reviews as positive or negative.  
Trained on ~16 000 reviews and evaluated on ~4 000, the model achieves a **93.8 % accuracy** and a **macro F1-score of 0.92**, demonstrating strong and well-balanced performance on real-world, noisy text data.

### Potential Improvements

| Idea | Expected Benefit |
|------|------------------|
| Add stop-word removal and lemmatisation | Cleaner vocabulary, reduced noise |
| Tune `C` (regularisation) via cross-validation | Better generalisation |
| Address class imbalance with `class_weight='balanced'` | Improve minority-class recall |
| Experiment with transformer models (e.g. DistilBERT) | Capture contextual and semantic meaning |
| Persist model with `joblib` | Enable production deployment |

---
## 10. Save Model — Deployment Ready

Once training is complete, both the **trained model** and the **fitted vectoriser** must be saved together.  
The vectoriser is just as important as the model — it converts raw text into the exact same numerical format the model was trained on.

We use Python's built-in `pickle` module to serialise both objects to disk.

| File | Contents | Purpose |
|------|----------|---------|
| `model.pkl` | Trained `LogisticRegression` | Makes predictions |
| `vectorizer.pkl` | Fitted `TfidfVectorizer` | Transforms new text before prediction |

> **Note:** To load them in a separate script or API, simply use `pickle.load()` — no retraining needed.

In [12]:
import pickle

# Save the trained model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)
print("model.pkl saved successfully.")

# Save the fitted vectoriser — required to transform new text at inference time
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("vectorizer.pkl saved successfully.")

model.pkl saved successfully.
vectorizer.pkl saved successfully.


In [14]:
# --- How to reload and use the saved model ---
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

print("Model and vectoriser loaded successfully.")

# Verify the loaded model produces the same result
test_text   = "Fast shipping and excellent product quality!"
cleaned     = clean_text(test_text)
vector      = loaded_vectorizer.transform([cleaned]).toarray()
prediction  = loaded_model.predict(vector)[0]
result      = "Positive" if prediction == 1 else "Negative"
icon        = "\u2705" if result == "Positive" else "\u274c"

print(f"Test prediction: {result} {icon}")

Model and vectoriser loaded successfully.
Test prediction: Positive ✅
